# Sales forecast: MLflow LightGBM model
Runs after `run_sales_forecast` (ai_forecast) so the app and Genie can compare both models. One global LightGBM model across all products forecasts weekly unit sales 12 weeks ahead from `gold.product_weekly_sales` (old + new GTIN combined). Features: sales 12-52 weeks earlier, a rolling mean, week of year, month, product and category (promotions are not used, since future promotions are unknown). Measures accuracy on the last 12 weeks, refits on all history, logs the model to MLflow and writes `gold.sales_forecast_mlflow`.

In [0]:
%pip install -q lightgbm
dbutils.library.restartPython()

In [0]:
import datetime as dt, warnings
import numpy as np, pandas as pd
import lightgbm as lgb, mlflow
from mlflow.models import infer_signature
warnings.filterwarnings("ignore")

dbutils.widgets.text("catalog", "patricia_aws_serverless_catalog")
CATALOG = dbutils.widgets.get("catalog")
H = 12
EXPERIMENT = "/Users/patricia.ferreiro@databricks.com/launch-forecast-comparison"

sales = spark.table(f"{CATALOG}.gold.product_weekly_sales").toPandas()
sales["week"] = pd.to_datetime(sales["week"])
today = pd.Timestamp(dt.date.today())
CUTOFF = min(sales.week.max(), today - pd.Timedelta(days=today.weekday()) - pd.Timedelta(weeks=1))
attrs = (sales.sort_values("week").groupby("product_key")
              .agg(client_id=("client_id", "first"), article_description=("article_description", "first"),
                   category=("category", "first"), current_gtin=("gtins_sold", "last")).reset_index())
hist = sales[sales.week <= CUTOFF][["product_key", "week", "units"]].rename(columns={"week": "ds", "units": "y"})
hist["y"] = hist["y"].astype(float)
hist = hist.merge(attrs[["product_key", "category"]], on="product_key")
PRODUCTS, CATEGORIES = sorted(hist.product_key.unique()), sorted(hist.category.unique())
print("cutoff", CUTOFF.date(), "|", len(PRODUCTS), "products |", hist.ds.nunique(), "weeks")

In [0]:
LAGS = [12, 13, 14, 26, 52]  # every lag >= horizon, so future rows only need known history
FEATURES = [f"lag_{l}" for l in LAGS] + ["roll_mean_12_19", "weekofyear", "month", "product", "cat"]
LGB_PARAMS = dict(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=10, verbose=-1, random_state=42)

def make_features(frame):
    frame, h = frame[["product_key", "category", "ds", "y"]].copy(), frame.dropna(subset=["y"])
    for l in LAGS:
        lagged = h[["product_key", "ds", "y"]].rename(columns={"y": f"lag_{l}"})
        lagged["ds"] = lagged["ds"] + pd.Timedelta(weeks=l)
        frame = frame.merge(lagged, on=["product_key", "ds"], how="left")
    roll = []
    for k, g in h.groupby("product_key"):
        r = g.set_index("ds")["y"].asfreq("W-MON").rolling(8, min_periods=4).mean()
        r.index = r.index + pd.Timedelta(weeks=12)  # mean of weeks t-19..t-12
        roll.append(pd.DataFrame({"product_key": k, "ds": r.index, "roll_mean_12_19": r.values}))
    frame = frame.merge(pd.concat(roll), on=["product_key", "ds"], how="left")
    frame["weekofyear"], frame["month"] = frame.ds.dt.isocalendar().week.astype(int), frame.ds.dt.month
    frame["product"] = pd.Categorical(frame.product_key, categories=PRODUCTS)
    frame["cat"] = pd.Categorical(frame.category, categories=CATEGORIES)
    return frame

def fit(train):
    f = make_features(train).dropna(subset=["lag_12"])
    models = {q: lgb.LGBMRegressor(objective="quantile", alpha=q, **LGB_PARAMS).fit(f[FEATURES], f["y"]) for q in (0.05, 0.95)}
    models["mean"] = lgb.LGBMRegressor(objective="regression", **LGB_PARAMS).fit(f[FEATURES], f["y"])
    return models, len(f)

class SalesForecaster(mlflow.pyfunc.PythonModel):
    """Takes future rows (product_key, ds = Monday of the week); returns forecast, forecast_lower, forecast_upper in units."""
    def __init__(self, models, history):
        self.models = models
        self.history = history.sort_values("ds").groupby("product_key").tail(max(LAGS) + 8)
        self.category = history.groupby("product_key").category.first().to_dict()
    def predict(self, context, model_input, params=None):
        future = model_input[["product_key", "ds"]].copy(); future["ds"] = pd.to_datetime(future["ds"])
        fr = make_features(pd.concat([self.history, future.assign(y=np.nan, category=future.product_key.map(self.category))], ignore_index=True))
        rows = fr[fr.y.isna()].copy()
        f, lo, hi = (np.clip(self.models[k].predict(rows[FEATURES]), 0, None) for k in ("mean", 0.05, 0.95))
        rows = rows[["product_key", "ds"]].assign(forecast=f, forecast_lower=np.minimum(lo, f), forecast_upper=np.maximum(hi, f))
        return future.merge(rows, on=["product_key", "ds"], how="left")

def future_rows(after):
    return pd.DataFrame([(p, t) for p in PRODUCTS for t in pd.date_range(after + pd.Timedelta(weeks=1), periods=H, freq="W-MON")],
                        columns=["product_key", "ds"])

In [0]:
# Holdout: train without the last 12 weeks, forecast them, compare with actual sales
h_start = CUTOFF - pd.Timedelta(weeks=H)
h_models, _ = fit(hist[hist.ds <= h_start])
h_pred = SalesForecaster(h_models, hist[hist.ds <= h_start]).predict(None, future_rows(h_start)).merge(hist, on=["product_key", "ds"])
e = (h_pred.y - h_pred.forecast)
acc = dict(model="mlflow_lightgbm", holdout_start=(h_start + pd.Timedelta(weeks=1)).date(), holdout_end=CUTOFF.date(),
           products=int(h_pred.product_key.nunique()), points=int(len(h_pred)),
           wape_pct=float(e.abs().sum() / h_pred.y.sum() * 100), mae_units=float(e.abs().mean()),
           coverage_pct=float(((h_pred.y >= h_pred.forecast_lower) & (h_pred.y <= h_pred.forecast_upper)).mean() * 100),
           bias_pct=float((h_pred.forecast.sum() - h_pred.y.sum()) / h_pred.y.sum() * 100))
print({k: round(v, 2) if isinstance(v, float) else v for k, v in acc.items()})

# Refit on all history and forecast the next 12 weeks
models, n_train = fit(hist)
model = SalesForecaster(models, hist)
future = future_rows(CUTOFF)
fc = model.predict(None, future)
assert len(fc) == len(PRODUCTS) * H and fc.forecast.notna().all(), "missing forecasts"

mlflow.set_experiment(EXPERIMENT)
with mlflow.start_run(run_name=f"sales_forecast_mlflow {CUTOFF.date()}") as run:
    mlflow.set_tags({"project": "product-launch-forecast", "target": "weekly unit sales per product", "method": "lightgbm_global"})
    mlflow.log_params({**{f"lgb_{k}": v for k, v in LGB_PARAMS.items()}, "lags": str(LAGS), "cutoff": str(CUTOFF.date()),
                       "horizon_weeks": H, "products": len(PRODUCTS), "n_train": n_train})
    mlflow.log_metrics({f"holdout_{k}": acc[k] for k in ("wape_pct", "mae_units", "coverage_pct", "bias_pct")})
    MODEL_URI = mlflow.pyfunc.log_model(name="model", python_model=model, signature=infer_signature(future, fc),
                                        input_example=future.head(5), extra_pip_requirements=[f"lightgbm=={lgb.__version__}"]).model_uri
    RUN_ID = run.info.run_id
loaded = mlflow.pyfunc.load_model(MODEL_URI).predict(future.head(24))
assert loaded.forecast.notna().all(), "logged model cannot forecast future rows"
print("run", RUN_ID, "| model", MODEL_URI)

In [0]:
now = pd.Timestamp.utcnow().tz_localize(None)
out = fc.merge(attrs, on="product_key").assign(model="mlflow_lightgbm", week=lambda x: x.ds.dt.date, generated_at=now, mlflow_run_id=RUN_ID)
out = out.rename(columns={"forecast": "forecast_units", "forecast_lower": "lower_units", "forecast_upper": "upper_units"})
out = out[["model", "product_key", "current_gtin", "client_id", "article_description", "category", "week",
           "forecast_units", "lower_units", "upper_units", "generated_at", "mlflow_run_id"]]
assert ((out.lower_units <= out.forecast_units) & (out.forecast_units <= out.upper_units) & (out.lower_units >= 0)).all()
spark.createDataFrame(out).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.sales_forecast_mlflow")
spark.sql(f"COMMENT ON TABLE {CATALOG}.gold.sales_forecast_mlflow IS 'MLflow LightGBM forecast of weekly unit sales per product (12 weeks, 90% range). Written by src/forecasting/run_sales_forecast_mlflow.'")

spark.sql(f"DELETE FROM {CATALOG}.gold.sales_forecast_accuracy WHERE model = 'mlflow_lightgbm'")
spark.createDataFrame(pd.DataFrame([{**acc, "products": acc["products"], "evaluated_at": now}])) \
     .select("model", "holdout_start", "holdout_end", "products", "points", "wape_pct", "mae_units", "coverage_pct", "bias_pct", "evaluated_at") \
     .write.mode("append").saveAsTable(f"{CATALOG}.gold.sales_forecast_accuracy")
display(spark.table(f"{CATALOG}.gold.sales_forecast_accuracy"))

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.sales_forecast (
  model COMMENT 'Forecast model: ai_forecast (Databricks SQL AI function) or mlflow_lightgbm (global LightGBM tracked in MLflow)',
  product_key COMMENT 'Product identifier: the original GTIN, kept across GTIN changeovers',
  current_gtin COMMENT 'GTIN(s) the product sold under in the latest week',
  client_id COMMENT 'Vendor / brand owner (e.g. NIVEA, Henkel, Unilever)',
  article_description COMMENT 'Product name',
  category COMMENT 'Product category',
  week COMMENT 'Monday of the forecast week',
  forecast_units COMMENT 'Forecast unit sales for the week, all stores combined',
  lower_units COMMENT 'Lower end of the 90% forecast range (units)',
  upper_units COMMENT 'Upper end of the 90% forecast range (units)',
  generated_at COMMENT 'When the forecast was produced'
)
COMMENT 'Weekly unit sales forecast per product for the next 12 weeks from two models. Compare models with gold.sales_forecast_accuracy (lower WAPE is better). Sales are synthetic demo data.'
AS
SELECT model, product_key, current_gtin, client_id, article_description, category, week, forecast_units, lower_units, upper_units, generated_at
FROM {CATALOG}.gold.sales_forecast_ai
UNION ALL
SELECT model, product_key, current_gtin, client_id, article_description, category, week, forecast_units, lower_units, upper_units, generated_at
FROM {CATALOG}.gold.sales_forecast_mlflow""")
display(spark.sql(f"SELECT model, COUNT(*) AS rows, COUNT(DISTINCT product_key) AS products, MIN(week) AS first, MAX(week) AS last FROM {CATALOG}.gold.sales_forecast GROUP BY model"))